<div style="background: linear-gradient(135deg, #7B4F00 0%, #f5a623 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #fff3d6; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #fff3d6; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: white; font-size: 1.05em; font-weight: 600;">&#9733; Bonus Chapter &nbsp;&middot;&nbsp; From Notebook to Production</span>
  </div>
</div>
<div style="background: #fff8ec; border-left: 5px solid #f5a623; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>Once a model is trained and saved, how do you make it available to other people?
  This chapter takes the trained models from Chapters 4 and 5 and deploys them as
  permanent, free web applications on Hugging Face Spaces &mdash; no server, no Docker,
  no command line required.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| B.1 The Gap Between Notebook and Production | Why a notebook is not enough · What a deployed model looks like |
| B.2 The Deployment Stack | Gradio · Hugging Face Spaces · How the three pieces fit together |
| B.3 Deploying the Churn Classifier (FFN) | Loading via ModelPipeline · Writing `app.py` · Uploading to Spaces · Testing the live app |
| B.4 Deploying the Defect Detector (CNN) | Image upload interface · PIL preprocessing · Live predictions |
| B.5 Updating a Deployed Model | Uploading a new `.pth` file · Changing one line · Rollback |

> **Prerequisites:** Chapter 4 (churn classifier) and Chapter 5 (defect detector CNN).
> The `.pth` files saved by `ModelPipeline.save()` in those chapters are what we deploy here.
>
> **Everything is free.** Hugging Face Spaces provides permanent free hosting.
> You only need a browser — no Docker, no terminal, no installation on your computer.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bonus Chapter — Setup
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet gradio torch torchvision scikit-learn dill numpy pillow

import torch, numpy as np, dill, gradio as gr
from PIL import Image

print(f'Gradio  : {gr.__version__}')
print(f'PyTorch : {torch.__version__}')
print('Setup complete.')

---
## Where We Are Coming From

In Chapters 4 and 5 you trained two models and saved them using `ModelPipeline.save()`:

```
churn_model_v1.pth      (Chapter 4 — FFN classifier: will a customer churn?)
defect_model_v1.pth     (Chapter 5 — CNN classifier: is this product defective?)
```

Each file is **self-contained**: the model class (serialised with `dill`), weights, scaler,
feature names, and training history — everything bundled together by `ModelPipeline.save()`.

**The problem:** only you can use them right now.

A product manager, a quality engineer, or a customer success team cannot open a
Jupyter notebook. They need a form they can fill in, or an image they can upload,
and a result they can act on.

**This chapter closes that gap — permanently and for free.**

By the end you will have two live web apps:

```
https://huggingface.co/spaces/your-username/churn-predictor
https://huggingface.co/spaces/your-username/defect-detector
```

Anyone with those links can use your models from any device, any browser, anywhere.
The apps stay live as long as your Hugging Face account exists.

> **Before starting:** make sure you have downloaded `churn_model_v1.pth` from your
> Chapter 4 session and `defect_model_v1.pth` from your Chapter 5 session.
> Each section in this chapter has an upload cell that walks you through this step.

---
# B.1 The Gap Between Notebook and Production

## What a notebook cannot do

A Jupyter notebook is a document. It requires a human to open it, run it, and read it.
No other person or program can send data to a notebook and get a result back automatically.

| | Notebook | Deployed app |
|-|----------|--------------|
| Who can use it | Only you, with the notebook open | Anyone with the URL |
| Device needed | Laptop with Python | Any browser — phone, tablet, PC |
| Requires technical knowledge | Yes | No |
| Always available | Only when your laptop is on | 24/7 |
| Another system can call it | No | Yes |

## What we are building

A non-technical user opens a URL and sees this:

```
Churn Predictor
───────────────────────────────────────────────
Gender                   [ Male ▼           ]
Senior Citizen           [ No  ▼            ]
Tenure (months)          [ 12        ]
Internet Service         [ Fiber optic ▼    ]
Contract                 [ Month-to-month ▼ ]
Monthly Charges ($)      [ 75.00     ]
Total Charges ($)        [ 900.00    ]
                         ... (all features)

                   [ Predict ]

Churn Probability:  68%
Risk Assessment:    High risk -- recommend a retention call immediately.
───────────────────────────────────────────────
```

They fill in the fields, click Predict, and get an answer.
No Python. No notebook. No setup. The model is running on Hugging Face's servers,
not on your laptop.

---
# B.2 The Deployment Stack

Three pieces work together. You only need to write the code — the other two
are free services that handle everything else.

```
┌──────────────────────────────────────────────────────────────┐
│                                                               │
│   churn_model_v1.pth    Gradio               HF Spaces       │
│   (ModelPipeline)  ──>  (web interface) ──>  (free hosting)  │
│                                                               │
│   Holds everything:      Turns your          Runs your app   │
│   weights, scaler,       predict_churn()     permanently     │
│   feature names,         function into       on the internet │
│   class definition       a web form                          │
└──────────────────────────────────────────────────────────────┘
```

---

## Gradio — you already know this

You used Gradio in Chapter 7 to build a chat interface.
Here you use it for a prediction form — the same idea, different input components.

```python
# Chapter 7 (chat):
gr.ChatInterface(fn=respond)

# Bonus Chapter (prediction form):
gr.Interface(fn=predict_churn, inputs=[...], outputs=[...])
```

`gr.Interface()` takes your prediction function and a list of input components
(one per feature) and builds the form automatically.

---

## Hugging Face Spaces — permanent free hosting

Hugging Face Spaces hosts Gradio apps for free, indefinitely.

To deploy, you upload exactly **three files** to a Space:

```
your-space/
    app.py               ← your Gradio application  (we write this in B.3 and B.4)
    churn_model_v1.pth   ← the pipeline file from Chapter 4
    requirements.txt     ← the Python packages the Space needs to install
```

Hugging Face reads those files, installs the packages, and runs `app.py`.
Your app is live at `https://huggingface.co/spaces/username/space-name`.

No command line. No Docker. No server configuration.
You upload files through a browser — like attaching files to an email.

---
# B.3 Deploying the Churn Classifier (FFN)

## Step 1 — Upload your model file

The churn model was saved as `churn_model_v1.pth` at the end of Chapter 4
using `pipeline.save('churn_model_v1.pth')`.

> **Where is the file?**
> If you ran Chapter 4 in Colab, the file was saved inside that session.
> Colab sessions do not share files — download it from the Chapter 4 session first,
> then upload it here.
>
> **To download from your Chapter 4 session:**
> Open that notebook in Colab. In the Files panel (folder icon, left sidebar),
> right-click `churn_model_v1.pth` and select **Download**.
> The file will appear in your browser's Downloads folder.
>
> **Then upload it here using the cell below.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload churn_model_v1.pth to this Colab session
#
# Click the "Choose Files" button that appears after running this cell.
# Select churn_model_v1.pth from your Downloads folder.
# Wait for the upload bar to complete before running any cells below.
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files
import os

print("Click 'Choose Files' and select churn_model_v1.pth")
print()

uploaded = files.upload()

if 'churn_model_v1.pth' in uploaded:
    size_mb = os.path.getsize('churn_model_v1.pth') / 1024 / 1024
    print(f'churn_model_v1.pth uploaded successfully  ({size_mb:.1f} MB)')
    print('You can now run the cells below.')
else:
    print('WARNING: the uploaded file is not named churn_model_v1.pth')
    print(f'  Files received: {list(uploaded.keys())}')
    print('  Rename the file to churn_model_v1.pth and re-run this cell.')

## Step 2 — Load the pipeline with ModelPipeline.load()

The `.pth` file contains everything: the model class (serialised with `dill`),
weights, the fitted `StandardScaler`, feature names, and training history.
`ModelPipeline.load()` reconstructs all of it in one call — exactly as in Chapter 4 §4.5.

No need to redefine `FFN`. No need to know the architecture. The class travels
inside the file.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the churn pipeline — identical to Chapter 4 §4.5
#
# ModelPipeline.load() reconstructs the FFN class from the dill bytes stored
# inside the .pth file. No class definition needed here.
# ─────────────────────────────────────────────────────────────────────────────

import copy, datetime
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import dill


class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Identical to the ModelPipeline in Chapter 3 §3.7 and Chapter 4 §4.5.
    Copied here so this notebook is self-contained.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor
        self.feature_names    = feature_names or []
        self.feature_ranges   = feature_ranges or {}
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {'train_loss': [], 'val_loss': []}

    def save(self, path):
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved -> {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  features    : {len(self.feature_names)}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    @classmethod
    def load(cls, path, device=None):
        device     = device or torch.device('cpu')
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        model_class = dill.loads(checkpoint['model_class_bytes'])
        ffn_keys = {'input_size', 'hidden_sizes', 'output_size', 'dropout_p'}
        ffn_cfg  = {k: v for k, v in checkpoint['model_config'].items() if k in ffn_keys}
        model    = model_class(**ffn_cfg)
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device)
        model.eval()
        pipeline = cls(
            model          = model,
            model_config   = checkpoint['model_config'],
            preprocessor   = checkpoint.get('preprocessor'),
            feature_names  = checkpoint.get('feature_names', []),
            feature_ranges = checkpoint.get('feature_ranges', {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            'training_history', {'train_loss': [], 'val_loss': []})
        print(f'Pipeline loaded <- {path}')
        print(f'  model_class    : {checkpoint.get("model_class")}  (reconstructed from dill)')
        print(f'  features       : {len(pipeline.feature_names)}')
        print(f'  history        : {len(pipeline.training_history["val_loss"])} epochs recorded')
        print(f'  pytorch_version: {checkpoint.get("pytorch_version")}')
        return pipeline

    def validate_input(self, X):
        if isinstance(X, pd.DataFrame):
            missing = set(self.feature_names) - set(X.columns)
            if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
            extra = set(X.columns) - set(self.feature_names)
            if extra: raise ValueError(f'Unexpected extra columns: {sorted(extra)}')
            null_cols = [c for c in self.feature_names if X[c].isnull().any()]
            if null_cols: raise ValueError(f'Null values in columns: {null_cols}')
        else:
            if X.ndim != 2: raise ValueError(f'Expected 2-D array, got shape {X.shape}')
            if self.feature_names and X.shape[1] != len(self.feature_names):
                raise ValueError(f'Expected {len(self.feature_names)} features, got {X.shape[1]}')
        return True

    def predict(self, X, device=None, threshold=0.5):
        """Validate, scale, and predict. Returns a labelled DataFrame.
        X must be a raw (unscaled) DataFrame with column names matching feature_names.
        The scaler is applied internally — identical to Chapter 4 §4.5.
        """
        device = device or torch.device('cpu')
        self.validate_input(X)
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].to_numpy()
        X = self.preprocessor.transform(X.astype(np.float32))
        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(
                self.model(torch.tensor(X, dtype=torch.float32).to(device)), dim=1
            )[:, 1].cpu().numpy()
        class_labels = self.model_config.get('class_labels', {0: '0', 1: '1'})
        return pd.DataFrame({
            'churn_probability': np.round(probs, 4),
            'predicted_label'  : [class_labels[int(p >= threshold)] for p in probs],
        })


print('ModelPipeline defined (source: Chapter 3 §3.7 / Chapter 4 §4.5).')

# ── Load the churn pipeline ───────────────────────────────────────────────────
# ModelPipeline.load() reconstructs the FFN class from the dill bytes
# stored inside churn_model_v1.pth. No class definition needed.
churn_pipeline = ModelPipeline.load('churn_model_v1.pth')

feature_names = churn_pipeline.feature_names
print(f'\nFeatures ({len(feature_names)}): {feature_names}')

## Step 3 — Write the Gradio predict function

The Gradio form will show dropdowns and sliders for every feature.
Our `predict_churn()` function:

1. Receives the raw string/numeric values the user typed into the form
2. Encodes categorical strings to integers — matching the `LabelEncoder` order used in Chapter 4
3. Assembles a **raw, unscaled** `pd.DataFrame` with the correct column names
4. Calls `churn_pipeline.predict()` — which applies the saved `StandardScaler` internally
5. Returns a probability string and a risk label

Step 4 is identical to the `loaded.predict(new_customers_raw)` call at the end of Chapter 4 §4.5.
The scaler is always applied inside `predict()` — never manually.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradio predict function for the churn classifier
#
# Categorical features are encoded to match the LabelEncoder alphabetical order
# used in Chapter 4 §4.3 preprocessing.
# The raw DataFrame is passed to churn_pipeline.predict() — the scaler is
# applied internally, exactly as in Chapter 4 §4.5.
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

# LabelEncoder alphabetical encoding — same order applied in Chapter 4 §4.3
ENCODE = {
    'gender':          {'Female': 0, 'Male': 1},
    'SeniorCitizen':   {'No': 0, 'Yes': 1},
    'Partner':         {'No': 0, 'Yes': 1},
    'Dependents':      {'No': 0, 'Yes': 1},
    'PhoneService':    {'No': 0, 'Yes': 1},
    'MultipleLines':   {'No': 0, 'No phone service': 1, 'Yes': 2},
    'InternetService': {'DSL': 0, 'Fiber optic': 1, 'No': 2},
    'OnlineSecurity':  {'No': 0, 'No internet service': 1, 'Yes': 2},
    'OnlineBackup':    {'No': 0, 'No internet service': 1, 'Yes': 2},
    'DeviceProtection':{'No': 0, 'No internet service': 1, 'Yes': 2},
    'TechSupport':     {'No': 0, 'No internet service': 1, 'Yes': 2},
    'StreamingTV':     {'No': 0, 'No internet service': 1, 'Yes': 2},
    'StreamingMovies': {'No': 0, 'No internet service': 1, 'Yes': 2},
    'Contract':        {'Month-to-month': 0, 'One year': 1, 'Two year': 2},
    'PaperlessBilling':{'No': 0, 'Yes': 1},
    'PaymentMethod':   {
        'Bank transfer (automatic)': 0,
        'Credit card (automatic)':   1,
        'Electronic check':          2,
        'Mailed check':              3,
    },
}


def predict_churn(gender, senior, partner, dependents, tenure,
                  phone, multiline, internet, security, backup,
                  device_prot, techsupport, tv, movies,
                  contract, paperless, payment, monthly, total):
    """
    Receives raw form values from Gradio.
    Encodes categoricals, builds a raw DataFrame, and calls
    churn_pipeline.predict() — which applies the StandardScaler internally.
    Returns (churn_probability_string, risk_label).
    """
    # Step 1: encode categoricals to match Chapter 4 LabelEncoder order
    row = {
        'gender'          : ENCODE['gender'][gender],
        'SeniorCitizen'   : ENCODE['SeniorCitizen'][senior],
        'Partner'         : ENCODE['Partner'][partner],
        'Dependents'      : ENCODE['Dependents'][dependents],
        'tenure'          : float(tenure),
        'PhoneService'    : ENCODE['PhoneService'][phone],
        'MultipleLines'   : ENCODE['MultipleLines'][multiline],
        'InternetService' : ENCODE['InternetService'][internet],
        'OnlineSecurity'  : ENCODE['OnlineSecurity'][security],
        'OnlineBackup'    : ENCODE['OnlineBackup'][backup],
        'DeviceProtection': ENCODE['DeviceProtection'][device_prot],
        'TechSupport'     : ENCODE['TechSupport'][techsupport],
        'StreamingTV'     : ENCODE['StreamingTV'][tv],
        'StreamingMovies' : ENCODE['StreamingMovies'][movies],
        'Contract'        : ENCODE['Contract'][contract],
        'PaperlessBilling': ENCODE['PaperlessBilling'][paperless],
        'PaymentMethod'   : ENCODE['PaymentMethod'][payment],
        'MonthlyCharges'  : float(monthly),
        'TotalCharges'    : float(total),
    }

    # Step 2: build a raw (unscaled) DataFrame — same as new_customers_raw in Ch4 §4.5
    df_input = pd.DataFrame([row])[feature_names]

    # Step 3: pipeline.predict() validates, scales, and runs the model internally
    result = churn_pipeline.predict(df_input)
    prob   = float(result['churn_probability'].iloc[0])

    # Step 4: format output
    if prob >= 0.70:
        risk = 'High risk  --  recommend a retention call immediately.'
    elif prob >= 0.40:
        risk = 'Medium risk  --  monitor and consider a proactive check-in.'
    else:
        risk = 'Low risk  --  customer appears stable.'

    return f'{prob:.0%}', risk


print('predict_churn() defined.')

# Sanity test — call directly before launching the UI
p, r = predict_churn(
    'Male', 'No', 'Yes', 'No', 12, 'Yes', 'No', 'Fiber optic',
    'No', 'No', 'No', 'No', 'No', 'No',
    'Month-to-month', 'Yes', 'Electronic check', 75.0, 900.0
)
print(f'Sanity test: {p}  |  {r}')

## Step 4 — Build the Gradio interface and test locally

Run this cell to see the churn app inside this notebook before uploading.
`share=False` keeps it local — no public URL yet.
Verify that the sanity test row produces the expected result before proceeding.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build the Gradio interface — local preview
#
# gr.Dropdown() for categorical features — choices match ENCODE dict keys.
# gr.Slider() / gr.Number() for numeric features.
# Two outputs: probability text + risk assessment text.
# ─────────────────────────────────────────────────────────────────────────────

demo = gr.Interface(
    fn = predict_churn,
    inputs = [
        gr.Dropdown(['Female', 'Male'],                                         label='Gender'),
        gr.Dropdown(['No', 'Yes'],                                              label='Senior Citizen'),
        gr.Dropdown(['No', 'Yes'],                                              label='Partner'),
        gr.Dropdown(['No', 'Yes'],                                              label='Dependents'),
        gr.Slider(0, 72, step=1, value=12,                                      label='Tenure (months)'),
        gr.Dropdown(['No', 'Yes'],                                              label='Phone Service'),
        gr.Dropdown(['No', 'No phone service', 'Yes'],                         label='Multiple Lines'),
        gr.Dropdown(['DSL', 'Fiber optic', 'No'],                              label='Internet Service'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Online Security'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Online Backup'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Device Protection'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Tech Support'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Streaming TV'),
        gr.Dropdown(['No', 'No internet service', 'Yes'],                      label='Streaming Movies'),
        gr.Dropdown(['Month-to-month', 'One year', 'Two year'],                label='Contract'),
        gr.Dropdown(['No', 'Yes'],                                              label='Paperless Billing'),
        gr.Dropdown(['Bank transfer (automatic)', 'Credit card (automatic)',
                     'Electronic check', 'Mailed check'],                      label='Payment Method'),
        gr.Slider(18, 120, step=0.5, value=65.0,                               label='Monthly Charges ($)'),
        gr.Number(value=780.0,                                                  label='Total Charges ($)'),
    ],
    outputs = [
        gr.Text(label='Churn Probability'),
        gr.Text(label='Risk Assessment'),
    ],
    title       = 'Customer Churn Predictor',
    description = 'Enter customer account details to predict churn likelihood.',
    examples    = [
        ['Male',   'No', 'Yes', 'No',  12, 'Yes', 'No',               'Fiber optic',
         'No', 'No', 'No', 'No', 'No', 'No',
         'Month-to-month', 'Yes', 'Electronic check',          75.0,  900.0],
        ['Female', 'No', 'Yes', 'Yes', 60, 'Yes', 'Yes',              'DSL',
         'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
         'Two year',       'No',  'Bank transfer (automatic)', 45.0, 2700.0],
        ['Male',   'Yes', 'No', 'No',  2,  'Yes', 'No phone service', 'Fiber optic',
         'No', 'No', 'No', 'No', 'Yes', 'Yes',
         'Month-to-month', 'Yes', 'Electronic check',          95.0,  190.0],
    ],
)

print('Launching local preview...')
demo.launch(share=False)

## Step 5 — Write app.py

Once the local preview looks correct, write `app.py`.
This file is an exact copy of the working code above — `ModelPipeline`,
`predict_churn()`, and the `gr.Interface()` call — bundled into a single
script that Hugging Face Spaces can run independently.

> **Why redefine `ModelPipeline` inside `app.py`?**
> HF Spaces runs `app.py` as a standalone script — there is no notebook,
> no shared session, no imports from other files. Everything the app needs
> must be inside `app.py`. `ModelPipeline` is small and has no side effects,
> so copying it in is the cleanest solution.
>
> The `FFN` class does **not** need to be copied — it is reconstructed
> automatically by `ModelPipeline.load()` from the `dill` bytes stored
> inside `churn_model_v1.pth`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write app.py for the churn classifier
#
# Contains: ModelPipeline (from Ch3/Ch4) + ENCODE dict + predict_churn() +
# gr.Interface() call. The FFN class is NOT redefined here — it travels
# inside churn_model_v1.pth as dill bytes and is reconstructed by .load().
# ─────────────────────────────────────────────────────────────────────────────

app_content = '''\
import copy, datetime
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import dill
import gradio as gr


# ── ModelPipeline ─────────────────────────────────────────────────────────────
# Source: Chapter 3 §3.7 / Chapter 4 §4.5
# Copied here so app.py is self-contained.
# The FFN class does NOT need to be redefined — it is reconstructed from the
# dill bytes stored inside churn_model_v1.pth by ModelPipeline.load().
class ModelPipeline:
    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor
        self.feature_names    = feature_names or []
        self.feature_ranges   = feature_ranges or {}
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {"train_loss": [], "val_loss": []}

    @classmethod
    def load(cls, path, device=None):
        device     = device or torch.device("cpu")
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        model_class = dill.loads(checkpoint["model_class_bytes"])
        ffn_keys = {"input_size", "hidden_sizes", "output_size", "dropout_p"}
        ffn_cfg  = {k: v for k, v in checkpoint["model_config"].items() if k in ffn_keys}
        model    = model_class(**ffn_cfg)
        model.load_state_dict(checkpoint["state_dict"])
        model.to(device)
        model.eval()
        pipeline = cls(
            model          = model,
            model_config   = checkpoint["model_config"],
            preprocessor   = checkpoint.get("preprocessor"),
            feature_names  = checkpoint.get("feature_names", []),
            feature_ranges = checkpoint.get("feature_ranges", {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            "training_history", {"train_loss": [], "val_loss": []})
        return pipeline

    def validate_input(self, X):
        if isinstance(X, pd.DataFrame):
            missing = set(self.feature_names) - set(X.columns)
            if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
            null_cols = [c for c in self.feature_names if X[c].isnull().any()]
            if null_cols: raise ValueError(f"Null values in columns: {null_cols}")
        return True

    def predict(self, X, device=None, threshold=0.5):
        device = device or torch.device("cpu")
        self.validate_input(X)
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].to_numpy()
        X = self.preprocessor.transform(X.astype(np.float32))
        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(
                self.model(torch.tensor(X, dtype=torch.float32).to(device)), dim=1
            )[:, 1].cpu().numpy()
        class_labels = self.model_config.get("class_labels", {0: "0", 1: "1"})
        return pd.DataFrame({
            "churn_probability": np.round(probs, 4),
            "predicted_label"  : [class_labels[int(p >= threshold)] for p in probs],
        })


# ── Load pipeline ─────────────────────────────────────────────────────────────
churn_pipeline = ModelPipeline.load("churn_model_v1.pth")
feature_names  = churn_pipeline.feature_names


# ── LabelEncoder encoding — matches Chapter 4 §4.3 preprocessing order ───────
ENCODE = {
    "gender":          {"Female": 0, "Male": 1},
    "SeniorCitizen":   {"No": 0, "Yes": 1},
    "Partner":         {"No": 0, "Yes": 1},
    "Dependents":      {"No": 0, "Yes": 1},
    "PhoneService":    {"No": 0, "Yes": 1},
    "MultipleLines":   {"No": 0, "No phone service": 1, "Yes": 2},
    "InternetService": {"DSL": 0, "Fiber optic": 1, "No": 2},
    "OnlineSecurity":  {"No": 0, "No internet service": 1, "Yes": 2},
    "OnlineBackup":    {"No": 0, "No internet service": 1, "Yes": 2},
    "DeviceProtection":{"No": 0, "No internet service": 1, "Yes": 2},
    "TechSupport":     {"No": 0, "No internet service": 1, "Yes": 2},
    "StreamingTV":     {"No": 0, "No internet service": 1, "Yes": 2},
    "StreamingMovies": {"No": 0, "No internet service": 1, "Yes": 2},
    "Contract":        {"Month-to-month": 0, "One year": 1, "Two year": 2},
    "PaperlessBilling":{"No": 0, "Yes": 1},
    "PaymentMethod":   {
        "Bank transfer (automatic)": 0,
        "Credit card (automatic)":   1,
        "Electronic check":          2,
        "Mailed check":              3,
    },
}


# ── Predict function ──────────────────────────────────────────────────────────
def predict_churn(gender, senior, partner, dependents, tenure,
                  phone, multiline, internet, security, backup,
                  device_prot, techsupport, tv, movies,
                  contract, paperless, payment, monthly, total):
    row = {
        "gender"          : ENCODE["gender"][gender],
        "SeniorCitizen"   : ENCODE["SeniorCitizen"][senior],
        "Partner"         : ENCODE["Partner"][partner],
        "Dependents"      : ENCODE["Dependents"][dependents],
        "tenure"          : float(tenure),
        "PhoneService"    : ENCODE["PhoneService"][phone],
        "MultipleLines"   : ENCODE["MultipleLines"][multiline],
        "InternetService" : ENCODE["InternetService"][internet],
        "OnlineSecurity"  : ENCODE["OnlineSecurity"][security],
        "OnlineBackup"    : ENCODE["OnlineBackup"][backup],
        "DeviceProtection": ENCODE["DeviceProtection"][device_prot],
        "TechSupport"     : ENCODE["TechSupport"][techsupport],
        "StreamingTV"     : ENCODE["StreamingTV"][tv],
        "StreamingMovies" : ENCODE["StreamingMovies"][movies],
        "Contract"        : ENCODE["Contract"][contract],
        "PaperlessBilling": ENCODE["PaperlessBilling"][paperless],
        "PaymentMethod"   : ENCODE["PaymentMethod"][payment],
        "MonthlyCharges"  : float(monthly),
        "TotalCharges"    : float(total),
    }
    df_input = pd.DataFrame([row])[feature_names]
    result   = churn_pipeline.predict(df_input)
    prob     = float(result["churn_probability"].iloc[0])
    if prob >= 0.70:
        risk = "High risk  --  recommend a retention call immediately."
    elif prob >= 0.40:
        risk = "Medium risk  --  monitor and consider a proactive check-in."
    else:
        risk = "Low risk  --  customer appears stable."
    return f"{prob:.0%}", risk


# ── Gradio interface ──────────────────────────────────────────────────────────
demo = gr.Interface(
    fn = predict_churn,
    inputs = [
        gr.Dropdown(["Female", "Male"],                                         label="Gender"),
        gr.Dropdown(["No", "Yes"],                                              label="Senior Citizen"),
        gr.Dropdown(["No", "Yes"],                                              label="Partner"),
        gr.Dropdown(["No", "Yes"],                                              label="Dependents"),
        gr.Slider(0, 72, step=1, value=12,                                      label="Tenure (months)"),
        gr.Dropdown(["No", "Yes"],                                              label="Phone Service"),
        gr.Dropdown(["No", "No phone service", "Yes"],                         label="Multiple Lines"),
        gr.Dropdown(["DSL", "Fiber optic", "No"],                              label="Internet Service"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Security"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Backup"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Device Protection"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Tech Support"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming TV"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming Movies"),
        gr.Dropdown(["Month-to-month", "One year", "Two year"],                label="Contract"),
        gr.Dropdown(["No", "Yes"],                                              label="Paperless Billing"),
        gr.Dropdown(["Bank transfer (automatic)", "Credit card (automatic)",
                     "Electronic check", "Mailed check"],                      label="Payment Method"),
        gr.Slider(18, 120, step=0.5, value=65.0,                               label="Monthly Charges ($)"),
        gr.Number(value=780.0,                                                  label="Total Charges ($)"),
    ],
    outputs = [
        gr.Text(label="Churn Probability"),
        gr.Text(label="Risk Assessment"),
    ],
    title       = "Customer Churn Predictor",
    description = "Enter customer account details to predict churn likelihood.",
    examples    = [
        ["Male",   "No", "Yes", "No",  12, "Yes", "No",               "Fiber optic",
         "No", "No", "No", "No", "No", "No",
         "Month-to-month", "Yes", "Electronic check",          75.0,  900.0],
        ["Female", "No", "Yes", "Yes", 60, "Yes", "Yes",              "DSL",
         "Yes", "Yes", "Yes", "Yes", "Yes", "Yes",
         "Two year",       "No",  "Bank transfer (automatic)", 45.0, 2700.0],
        ["Male",   "Yes", "No", "No",  2,  "Yes", "No phone service", "Fiber optic",
         "No", "No", "No", "No", "Yes", "Yes",
         "Month-to-month", "Yes", "Electronic check",          95.0,  190.0],
    ],
)

if __name__ == "__main__":
    demo.launch()
'''

with open('app.py', 'w') as f:
    f.write(app_content)

print('app.py written.')
print(f'  Lines : {len(app_content.splitlines())}')
print('To download: right-click app.py in the Colab file browser and select Download.')

## Step 6 — Create requirements.txt and README.md

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# requirements.txt
#
# dill is required — ModelPipeline.load() uses it to reconstruct the FFN class.
# scikit-learn must match the version used to fit the StandardScaler in Ch4.
# ─────────────────────────────────────────────────────────────────────────────

req_lines = [
    'torch==2.2.0',
    'numpy==1.26.4',
    'scikit-learn==1.6.1',
    'pandas==2.2.2',
    'dill==0.3.8',
]

with open('requirements.txt', 'w') as f:
    f.write('\n'.join(req_lines) + '\n')

print('requirements.txt written:')
for line in req_lines:
    print(f'  {line}')
print('  (gradio installed automatically by HF Spaces)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# README.md
#
# python_version: "3.10" avoids the Python 3.13 torch segfault.
# Do NOT set sdk_version — let HF Spaces use its own compatible gradio.
# ─────────────────────────────────────────────────────────────────────────────

readme_content = """---
title: Customer Churn Predictor
emoji: \U0001f4ca
colorFrom: blue
colorTo: green
sdk: gradio
app_file: app.py
pinned: false
python_version: "3.10"
---

# Customer Churn Predictor
Predicts the likelihood a telecom customer will cancel their subscription.
Model trained on the Telco Customer Churn dataset (Chapter 4).
Pipeline saved with ModelPipeline (Chapter 3 §3.7 / Chapter 4 §4.5).
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print('README.md written.')
print('Key settings:')
print('  python_version: 3.10  (avoids torch segfault on Python 3.13)')
print('  sdk_version: not set  (HF Spaces installs its own compatible gradio)')

## Step 7 — Upload to Hugging Face Spaces

Once the local preview looks correct, uploading takes about 5 minutes.

---

### 1. Create a free Hugging Face account

Go to [huggingface.co](https://huggingface.co) and sign up.
No credit card. No personal information required beyond an email address.

---

### 2. Create a new Space

1. Click your profile picture → **New Space**
2. Fill in the form:

| Field | Value |
|-------|-------|
| Space name | `churn-predictor` |
| **SDK** | **Gradio** ← important |
| Hardware | CPU Basic (free) |
| Visibility | Public or Private |

3. Click **Create Space**

---

### 3. Upload your four files

Click **Add file → Upload files** and upload:

```
app.py
churn_model_v1.pth
requirements.txt
README.md
```

> Download all four from the Colab Files panel (folder icon, left sidebar):
> right-click each file and select **Download**.

---

### 4. Watch the build and get your URL

After uploading, Hugging Face automatically:
1. Installs packages from `requirements.txt` (2–4 minutes)
2. Runs `app.py`
3. Shows your live app at `https://huggingface.co/spaces/your-username/churn-predictor`

When the build finishes you will see a green **Running** badge.

> **If the build fails**, the Logs tab shows the error. The most common cause is
> a version mismatch in `requirements.txt`. Check that `dill` and `scikit-learn`
> versions match what was used to save the `.pth` file.

---
# B.4 Deploying the Defect Detector (CNN)

The CNN from Chapter 5 takes an **image** as input instead of a set of numbers.
Gradio handles image uploads natively — `gr.Image()` gives the user an upload
button, and Gradio passes the uploaded image directly to your function as a
PIL Image object.

The only difference from B.3:
- Input component: `gr.Image(type='pil')` instead of dropdowns and sliders
- `predict_defect()` applies the image transforms itself — the `ModelPipeline`
  preprocessor is `None` for the CNN (images do not use `StandardScaler`)
- Transforms must **exactly match** the training transforms from Chapter 5

Everything else — loading, `app.py`, `requirements.txt`, HF Spaces upload — is identical to B.3.

## Before you start — upload your model file

The defect detector was saved as `defect_model_v1.pth` at the end of Chapter 5.
Download it from your Chapter 5 session (Files panel → right-click → Download),
then upload it here.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload defect_model_v1.pth to this Colab session
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files
import os

print("Click 'Choose Files' and select defect_model_v1.pth")
print()

uploaded = files.upload()

if 'defect_model_v1.pth' in uploaded:
    size_mb = os.path.getsize('defect_model_v1.pth') / 1024 / 1024
    print(f'defect_model_v1.pth uploaded successfully  ({size_mb:.1f} MB)')
    print('You can now run the cells below.')
else:
    print('WARNING: the uploaded file is not named defect_model_v1.pth')
    print(f'  Files received: {list(uploaded.keys())}')
    print('  Rename the file to defect_model_v1.pth and re-run this cell.')

## Step 2 — Load the pipeline and test locally

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the defect detector and build the Gradio interface
#
# ModelPipeline.load() reconstructs the DefectCNN class from the dill bytes
# stored inside defect_model_v1.pth — no class definition needed.
#
# Image preprocessing: must match the training transforms in Chapter 5 exactly.
# The StandardScaler is not used for images — pipeline.preprocessor is None.
# ─────────────────────────────────────────────────────────────────────────────

from torchvision import transforms
from PIL import Image
import gradio as gr

# Load the defect pipeline — same call as Chapter 5's ModelPipeline.load()
defect_pipeline = ModelPipeline.load('defect_model_v1.pth')
CLASS_NAMES     = defect_pipeline.model_config.get('class_names', {0: 'Defective', 1: 'OK'})
print(f'Model loaded. Classes: {CLASS_NAMES}')

# ── Image transforms — must match Chapter 5 training transforms exactly ───────
TRANSFORM = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


def predict_defect(image):
    """
    Receives a PIL Image from the Gradio upload component.
    Applies the same transforms used during Chapter 5 training.
    Runs the model directly (no StandardScaler — images don't use one).
    Returns (verdict_string, confidence_scores_dict).
    """
    if image is None:
        return 'No image uploaded.', {}

    tensor = TRANSFORM(image).unsqueeze(0)   # (1, 1, 64, 64)

    with torch.no_grad():
        logits = defect_pipeline.model(tensor)
        probs  = torch.softmax(logits, dim=1).numpy()[0]

    pred_idx   = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])

    if pred_label == 'Defective' and confidence >= 0.80:
        verdict = f'DEFECT DETECTED  ({confidence:.0%} confidence)  -- flag for inspection.'
    elif pred_label == 'Defective':
        verdict = f'Possible defect  ({confidence:.0%} confidence)  -- manual check recommended.'
    else:
        verdict = f'No defect  ({confidence:.0%} confidence)  -- product OK.'

    scores = {str(CLASS_NAMES[i]): float(probs[i]) for i in range(len(probs))}
    return verdict, scores


# Sanity test with a blank image
dummy = Image.fromarray((np.ones((64, 64, 3)) * 128).astype(np.uint8))
v, s  = predict_defect(dummy)
print(f'Sanity test verdict: {v}')

# ── Gradio interface ──────────────────────────────────────────────────────────
defect_demo = gr.Interface(
    fn      = predict_defect,
    inputs  = gr.Image(type='pil', label='Upload product image'),
    outputs = [
        gr.Text(label='Verdict'),
        gr.Label(label='Confidence scores'),
    ],
    title       = 'Product Defect Detector',
    description = 'Upload a product image to classify it as OK or defective. Model trained on casting defect images (Chapter 5).',
)

print('Launching local preview...')
defect_demo.launch(share=False)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write app.py for the defect detector
#
# Contains: ModelPipeline (from Ch3/Ch4) + TRANSFORM + predict_defect() +
# gr.Interface() call. DefectCNN is NOT redefined — it travels inside
# defect_model_v1.pth as dill bytes, reconstructed by ModelPipeline.load().
# ─────────────────────────────────────────────────────────────────────────────

defect_app_content = '''\
import torch
import numpy as np
import pandas as pd
import dill
import gradio as gr
from torchvision import transforms
from PIL import Image


# ── ModelPipeline ─────────────────────────────────────────────────────────────
# Source: Chapter 3 §3.7 / Chapter 4 §4.5  (copied for self-contained app.py)
class ModelPipeline:
    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model         = model
        self.model_config  = model_config
        self.preprocessor  = preprocessor
        self.feature_names = feature_names or []
        self._model_class  = model_class
        self.model_class   = model_class.__name__ if model_class else type(model).__name__

    @classmethod
    def load(cls, path, device=None):
        device      = device or torch.device("cpu")
        checkpoint  = torch.load(path, map_location=device, weights_only=False)
        model_class = dill.loads(checkpoint["model_class_bytes"])
        cfg_keys    = set(model_class.__init__.__code__.co_varnames)
        cfg         = {k: v for k, v in checkpoint["model_config"].items() if k in cfg_keys}
        model       = model_class(**cfg)
        model.load_state_dict(checkpoint["state_dict"])
        model.to(device)
        model.eval()
        return cls(
            model          = model,
            model_config   = checkpoint["model_config"],
            preprocessor   = checkpoint.get("preprocessor"),
            feature_names  = checkpoint.get("feature_names", []),
            model_class    = model_class,
        )


# ── Load pipeline ─────────────────────────────────────────────────────────────
defect_pipeline = ModelPipeline.load("defect_model_v1.pth")
CLASS_NAMES     = defect_pipeline.model_config.get("class_names", {0: "Defective", 1: "OK"})


# ── Image transforms — must match Chapter 5 training transforms exactly ───────
TRANSFORM = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


# ── Predict function ──────────────────────────────────────────────────────────
def predict_defect(image):
    if image is None:
        return "No image uploaded.", {}
    tensor = TRANSFORM(image).unsqueeze(0)
    with torch.no_grad():
        logits = defect_pipeline.model(tensor)
        probs  = torch.softmax(logits, dim=1).numpy()[0]
    pred_idx   = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])
    if pred_label == "Defective" and confidence >= 0.80:
        verdict = f"DEFECT DETECTED  ({confidence:.0%} confidence)  -- flag for inspection."
    elif pred_label == "Defective":
        verdict = f"Possible defect  ({confidence:.0%} confidence)  -- manual check recommended."
    else:
        verdict = f"No defect  ({confidence:.0%} confidence)  -- product OK."
    scores = {str(CLASS_NAMES[i]): float(probs[i]) for i in range(len(probs))}
    return verdict, scores


# ── Gradio interface ──────────────────────────────────────────────────────────
demo = gr.Interface(
    fn      = predict_defect,
    inputs  = gr.Image(type="pil", label="Upload product image"),
    outputs = [
        gr.Text(label="Verdict"),
        gr.Label(label="Confidence scores"),
    ],
    title       = "Product Defect Detector",
    description = "Upload a product image to classify it as OK or defective. Model trained on casting defect images (Chapter 5).",
)

if __name__ == "__main__":
    demo.launch()
'''

with open('app.py', 'w') as f:
    f.write(defect_app_content)

print('app.py written for defect detector.')
print(f'  Lines : {len(defect_app_content.splitlines())}')
print('To download: right-click app.py in the Colab file browser and select Download.')

In [ ]:
# requirements.txt for the defect detector Space
# torchvision added; dill required for ModelPipeline.load()

req_lines = [
    'torch==2.2.0',
    'torchvision==0.17.0',
    'numpy==1.26.4',
    'dill==0.3.8',
    'pillow==10.3.0',
]

with open('requirements.txt', 'w') as f:
    f.write('\n'.join(req_lines) + '\n')

print('requirements.txt written:')
for line in req_lines:
    print(f'  {line}')
print('  (gradio installed automatically by HF Spaces)')

In [ ]:
readme_content = """---
title: Product Defect Detector
emoji: \U0001f50d
colorFrom: red
colorTo: red
sdk: gradio
app_file: app.py
pinned: false
python_version: "3.10"
---

# Product Defect Detector
Classifies product images as OK or defective.
Model trained on the Casting Defect dataset (Chapter 5).
Pipeline saved with ModelPipeline (Chapter 3 §3.7 / Chapter 4 §4.5).
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print('README.md written.')

---
### 📝 Exercise B.4 — Deploy both apps

1. Deploy the churn predictor to a Space named `churn-predictor` (Section B.3 steps).
2. Deploy the defect detector to a Space named `defect-detector` (same steps, different files).
3. Open both live URLs and test them:
   - For the churn predictor: try the three example rows. Do the risk levels match your intuition?
   - For the defect detector: upload one of the casting defect images from the Chapter 5 dataset. Does the model classify it correctly?
4. Share one URL with someone who has not seen the notebook. Can they use the app without any explanation from you?

---
# B.5 Updating a Deployed Model

When you retrain a model with `pipeline.retrain()` and save a new `.pth` file
using `pipeline.save()`, updating the live app is a two-step process that takes
about 2 minutes.

## The update workflow

```
Notebook                          Hugging Face Space
────────                          ──────────────────
pipeline.retrain(...)             |
     |                            |
pipeline.save('churn_model_v2.pth')
     |                            |
go to Space in browser  ──────>  Files tab
                                      |
upload churn_model_v2.pth        replaces v1.pth
update app.py (one line)         rebuilds automatically
                                      |
test live URL          <──────── app is live with v2
```

---

## What changes in `app.py`

Exactly one line:

```python
# Before
churn_pipeline = ModelPipeline.load('churn_model_v1.pth')

# After
churn_pipeline = ModelPipeline.load('churn_model_v2.pth')
```

Upload the new `.pth` file and the updated `app.py` through the Files tab.
Hugging Face detects the change and rebuilds automatically.

---

## Version naming — same discipline as Chapters 3–4

| File | When |
|------|------|
| `churn_model_v1.pth` | Initial training (Chapter 4) |
| `churn_model_v2.pth` | Retrained on new monthly data |
| `churn_model_v3.pth` | Architecture change or major dataset update |

Never delete old `.pth` files from your Space. If `v2` performs worse than `v1`,
you roll back by uploading the old `app.py` that points to `v1.pth`.
One file upload. Done.

> **The same versioning discipline practised since Chapter 3 applies here.**
> Every `.pth` file is a checkpoint. Every version is a rollback option.

---
## Bonus Chapter Summary

| Concept | Key takeaway |
|---------|-------------|
| **The gap** | A notebook requires a human; a deployed app serves anyone automatically, 24/7 |
| **ModelPipeline.load()** | Reconstructs the model class from `dill` bytes — no class definition needed in `app.py` |
| **pipeline.predict()** | Validates input, applies the saved `StandardScaler`, runs the model — same call as Chapter 4 §4.5 |
| **ENCODE dict** | Converts UI dropdown strings to integers matching the Chapter 4 `LabelEncoder` order |
| **Gradio** | `gr.Interface()` turns `predict_churn()` into a web form automatically |
| **Input components** | `gr.Dropdown()` for categoricals; `gr.Slider()` / `gr.Number()` for numerics; `gr.Image()` for CNN inputs |
| **Image preprocessing** | The `TRANSFORM` pipeline in `predict_defect()` must match Chapter 5 training transforms exactly |
| **HF Spaces** | Upload `app.py`, `.pth`, `requirements.txt`, `README.md` → permanent free URL |
| **requirements.txt** | `dill` is required — `ModelPipeline.load()` uses it to reconstruct the model class |
| **Updating** | `pipeline.save('v2.pth')` → upload → change one line in `app.py` → Space rebuilds |
| **Rollback** | Re-upload the old `app.py` pointing to the previous `.pth` — done in one file upload |

---

## The Complete Journey

```
Chapter 1  -->  A single neuron learns to fit a line
Chapter 2  -->  NumPy, pandas, PyTorch tensors
Chapter 3  -->  Training, saving, loading: the ModelPipeline
Chapter 4  -->  FFN: customer churn classifier
Chapter 5  -->  CNN: product defect detector
Chapter 6  -->  LSTM: CO2 demand forecaster
Chapter 7  -->  LLMs: earnings analyst bot
Bonus      -->  Sharing models as live web apps on Hugging Face Spaces
```

You started with a single artificial neuron. You finish with two live web apps
that anyone in the world can use from their browser.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam & Mr. Daniel Peter*